## Association Mining with FP-Growth

This notebook demonstrates how to use the FP-Growth algorithm to find frequent itemsets and association rules in the flight delay dataset.

In [169]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv('../data/cleaned_flight_data.csv')
df.head()

,scheduled_time,airline,flight_number,destination_or_origin,status,counter,actual_time,register,aircraft,airport,...,scheduled_day_of_week,is_weekend,is_Nourooz_4,is_Nourooz_13,Normal_holiday,scheduled_season,scheduled_time_of_day,Early_OnTime_Late_Indicator,Actual_Day_Matches_Scheduled_Day,Scheduled_Hour_of_Day
0,2025-05-28 06:50:00,کاسپین,CPN024,مشهد,پرواز كرد,"12, 13",2025-05-28 07:21:00,EPCPU,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,6
1,2025-05-28 07:00:00,اطلس ایر,ATS8271,كيش,پرواز كرد,"16, 17",2025-05-28 07:39:00,EPSAP,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
2,2025-05-28 07:00:00,آتا,TBZ5700,سیرجان,پرواز كرد,"26, 27",2025-05-28 07:27:00,EPTAH,737-700,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
3,2025-05-28 07:05:00,پويا,PYA2350,اصفهان,پرواز كرد,14,2025-05-28 07:23:00,EPPUN,EMB145,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
4,2025-05-28 07:05:00,ایران ایر,IRA311,اهواز,پرواز كرد,"5, 6",2025-05-28 07:29:00,EPIEQ,A319,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7


In [171]:
air = df[df['airport'] == 'فرودگاه مهرآباد'].copy()
print(f"percentage: {(air.shape[0])/df.shape[0]}")

percentage: 0.9148292261132728


In [198]:
df.columns

Index(['scheduled_time', 'airline', 'flight_number', 'destination_or_origin',
       'status', 'counter', 'actual_time', 'register', 'aircraft', 'airport',
       'flight_type', 'scheduled', 'actual', 'scheduled_datetime',
       'actual_datetime', 'delay', 'delay_minutes', 'scheduled_day_of_week',
       'is_weekend', 'is_Nourooz_4', 'is_Nourooz_13', 'Normal_holiday',
       'scheduled_season', 'scheduled_time_of_day',
       'Early_OnTime_Late_Indicator', 'Actual_Day_Matches_Scheduled_Day',
       'Scheduled_Hour_of_Day'],
      dtype='object')

In [172]:
# Select features for association mining
# df_assoc = df[['airline', 'destination_or_origin', 'aircraft', 'airport', 'scheduled_day_of_week', 'scheduled_season', 'scheduled_time_of_day', 'Early_OnTime_Late_Indicator']].copy()
df_assoc = air[['airline', 'destination_or_origin', 'aircraft', 'scheduled_day_of_week', 'scheduled_season', 'scheduled_time_of_day', 'Early_OnTime_Late_Indicator']].copy()
# Drop rows with missing values
df_assoc.dropna(inplace=True)

df.head()

,scheduled_time,airline,flight_number,destination_or_origin,status,counter,actual_time,register,aircraft,airport,...,scheduled_day_of_week,is_weekend,is_Nourooz_4,is_Nourooz_13,Normal_holiday,scheduled_season,scheduled_time_of_day,Early_OnTime_Late_Indicator,Actual_Day_Matches_Scheduled_Day,Scheduled_Hour_of_Day
0,2025-05-28 06:50:00,کاسپین,CPN024,مشهد,پرواز كرد,"12, 13",2025-05-28 07:21:00,EPCPU,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,6
1,2025-05-28 07:00:00,اطلس ایر,ATS8271,كيش,پرواز كرد,"16, 17",2025-05-28 07:39:00,EPSAP,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
2,2025-05-28 07:00:00,آتا,TBZ5700,سیرجان,پرواز كرد,"26, 27",2025-05-28 07:27:00,EPTAH,737-700,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
3,2025-05-28 07:05:00,پويا,PYA2350,اصفهان,پرواز كرد,14,2025-05-28 07:23:00,EPPUN,EMB145,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
4,2025-05-28 07:05:00,ایران ایر,IRA311,اهواز,پرواز كرد,"5, 6",2025-05-28 07:29:00,EPIEQ,A319,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7


In [173]:
# Convert the dataframe into a list of transactions
transactions = df_assoc.to_numpy().tolist()

In [174]:
# Use TransactionEncoder to transform the data into a one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

In [175]:
# Run the FP-Growth algorithm
frequent_itemsets = fpgrowth(df_onehot, min_support=0.01, use_colnames=True)

In [176]:
# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Display the rules
rules.sort_values(by='lift', ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
7763,(MD.88),"(Spring, تابان)",0.015595,0.011815,0.011815,0.757576,64.121212,1.0,0.01163,4.076264,1.000000,0.757576,0.754677,0.878788
7759,"(Spring, MD.88)",(تابان),0.015595,0.011815,0.011815,0.757576,64.121212,1.0,0.01163,4.076264,1.000000,0.757576,0.754677,0.878788
7758,"(Spring, تابان)",(MD.88),0.011815,0.015595,0.011815,1.000000,64.121212,1.0,0.01163,inf,0.996174,0.757576,1.000000,0.878788
7755,(MD.88),(تابان),0.015595,0.011815,0.011815,0.757576,64.121212,1.0,0.01163,4.076264,1.000000,0.757576,0.754677,0.878788
7762,(تابان),"(Spring, MD.88)",0.011815,0.015595,0.011815,1.000000,64.121212,1.0,0.01163,inf,0.996174,0.757576,1.000000,0.878788


In [177]:
late_rules = rules[rules['consequents'].apply(lambda x: 'Late' in x)].copy()
late_rules.shape

(2206, 14)

In [178]:
late_rules.sort_values(by='confidence', ascending=False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
196,"(Spring, MD83, Evening, مشهد)",(Late),0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
203,"(MD83, Evening, مشهد)","(Spring, Late)",0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
185,"(MD83, Evening, مشهد)",(Late),0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
6900,"(Spring, MD83, Sunday, آتا)",(Late),0.015123,0.6569,0.014178,0.937500,1.427158,1.0,0.004243,5.489603,0.303903,0.021552,0.817837,0.479541
6911,"(MD83, آتا, Sunday)","(Spring, Late)",0.015123,0.6569,0.014178,0.937500,1.427158,1.0,0.004243,5.489603,0.303903,0.021552,0.817837,0.479541


In [179]:
late_rules_freeze = rules[rules['consequents'] == frozenset({'Late'})].copy()
late_rules_freeze.shape

(333, 14)

In [180]:
late_rules_freeze = late_rules_freeze[late_rules_freeze['antecedents'].apply(lambda x: 'Spring' not in x)].copy()
late_rules_freeze.shape

(166, 14)

In [181]:
late_rules_freeze.sort_values(by='confidence', ascending= False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
185,"(MD83, Evening, مشهد)",(Late),0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
6886,"(MD83, آتا, Sunday)",(Late),0.015123,0.6569,0.014178,0.937500,1.427158,1.0,0.004243,5.489603,0.303903,0.021552,0.817837,0.479541
1754,"(MD83, آتا, Evening)",(Late),0.024102,0.6569,0.022212,0.921569,1.402906,1.0,0.006379,4.374527,0.294287,0.033716,0.771404,0.477691
2254,"(MD83, آتا, اهواز)",(Late),0.011342,0.6569,0.010397,0.916667,1.395444,1.0,0.002946,4.117202,0.286633,0.015805,0.757117,0.466247
1726,"(آتا, Evening)",(Late),0.034026,0.6569,0.031191,0.916667,1.395444,1.0,0.008839,4.117202,0.293364,0.047278,0.757117,0.482074


In [182]:
late_rules_freeze['antecedents_len'] = late_rules_freeze['antecedents'].apply(lambda x: len(x))

In [183]:
late_rules_freeze.sort_values(by=['antecedents_len'], ascending =[True]).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
8,(MD83),(Late),0.314272,0.6569,0.233459,0.742857,1.130853,1.0,0.027014,1.334279,0.168743,0.316464,0.250531,0.549126,1
45,(مشهد),(Late),0.226843,0.6569,0.166352,0.733333,1.116355,1.0,0.017338,1.286626,0.134808,0.231884,0.222773,0.493285,1
225,(Wednesday),(Late),0.198488,0.6569,0.138941,0.700000,1.065612,1.0,0.008555,1.143667,0.076819,0.193931,0.125620,0.455755,1
806,(كيش),(Late),0.083648,0.6569,0.055293,0.661017,1.006268,1.0,0.000344,1.012146,0.006797,0.080690,0.012000,0.372595,1
1058,(اطلس ایر),(Late),0.033554,0.6569,0.023629,0.704225,1.072044,1.0,0.001588,1.160005,0.069535,0.035436,0.137935,0.370098,1


## **By day**

In [184]:
days_of_the_week = set(df['scheduled_day_of_week'].unique())

def only_things(antecedents, _set):
    return antecedents.issubset(_set)

In [185]:
late_rules_by_day = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (days_of_the_week, ))].copy()
late_rules_by_day.shape
late_rules_by_day

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
225,(Wednesday),(Late),0.198488,0.6569,0.138941,0.700000,1.065612,1.0,0.008555,1.143667,0.076819,0.193931,0.125620,0.455755,1
8162,(Monday),(Late),0.140832,0.6569,0.095463,0.677852,1.031896,1.0,0.002951,1.065040,0.035977,0.135935,0.061068,0.411588,1
8716,(Tuesday),(Late),0.126181,0.6569,0.093573,0.741573,1.128898,1.0,0.010684,1.327649,0.130669,0.135709,0.246789,0.442010,1


## **By destination**

In [186]:
destinations = set(df['destination_or_origin'].unique())

In [187]:
late_rules_by_dest = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (destinations, ))].copy()
late_rules_by_dest.shape

(9, 15)

In [188]:
late_rules_by_dest.sort_values(by='confidence', ascending = False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
7168,(آبادان),(Late),0.022212,0.6569,0.017958,0.808511,1.230797,1.0,0.003368,1.791745,0.191778,0.027162,0.441885,0.417924,1
2148,(اهواز),(Late),0.068526,0.6569,0.054348,0.793103,1.207343,1.0,0.009333,1.658318,0.184369,0.080986,0.396979,0.437919,1
45,(مشهد),(Late),0.226843,0.6569,0.166352,0.733333,1.116355,1.0,0.017338,1.286626,0.134808,0.231884,0.222773,0.493285,1
4870,(بندرعباس),(Late),0.038280,0.6569,0.026465,0.691358,1.052456,1.0,0.001319,1.111645,0.051825,0.039576,0.100432,0.365823,1
6348,(تبریز),(Late),0.049622,0.6569,0.034026,0.685714,1.043864,1.0,0.001430,1.091682,0.044215,0.050597,0.083983,0.368756,1


## **by Airlines only**

In [189]:
airlines = set(df['airline'].unique())

In [190]:
late_rules_by_airline = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (airlines, ))].copy()
late_rules_by_airline.shape

(8, 15)

In [191]:
late_rules_by_airline

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
1058,(اطلس ایر),(Late),0.033554,0.6569,0.023629,0.704225,1.072044,1.0,0.001588,1.160005,0.069535,0.035436,0.137935,0.370098,1
1098,(آتا),(Late),0.152647,0.6569,0.125709,0.823529,1.253661,1.0,0.025435,1.944234,0.238786,0.183829,0.485659,0.507448,1
4072,(وارش),(Late),0.050095,0.6569,0.037807,0.754717,1.148907,1.0,0.004900,1.398793,0.136443,0.056497,0.285098,0.406135,1
4916,(آسمان),(Late),0.028828,0.6569,0.021267,0.737705,1.123010,1.0,0.002329,1.308069,0.112787,0.032006,0.235515,0.385040,1
5184,(کیش ایر),(Late),0.063327,0.6569,0.043478,0.686567,1.045163,1.0,0.001879,1.094653,0.046133,0.064246,0.086468,0.376377,1
5636,(قشم ایر),(Late),0.051985,0.6569,0.037335,0.718182,1.093290,1.0,0.003186,1.217452,0.090008,0.055595,0.178613,0.387508,1
5814,(ایران ایرتور),(Late),0.046314,0.6569,0.032609,0.704082,1.071825,1.0,0.002185,1.159442,0.070266,0.048626,0.137516,0.376861,1
7640,(زاگرس),(Late),0.037335,0.6569,0.033081,0.886076,1.348875,1.0,0.008556,3.011657,0.268672,0.050036,0.667957,0.468218,1


### **by Aircraft**

In [192]:
aircrafts = set(df['aircraft'].unique())

In [193]:
late_rules_by_aircraft = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (aircrafts, ))].copy()
late_rules_by_aircraft.shape

(6, 15)

In [194]:
late_rules_by_aircraft

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
8,(MD83),(Late),0.314272,0.6569,0.233459,0.742857,1.130853,1.0,0.027014,1.334279,0.168743,0.316464,0.250531,0.549126,1
3156,(rj100),(Late),0.025520,0.6569,0.017486,0.685185,1.043059,1.0,0.000722,1.089848,0.042362,0.026297,0.082441,0.355902,1
3740,(B737),(Late),0.092155,0.6569,0.066635,0.723077,1.100742,1.0,0.006099,1.238973,0.100812,0.097645,0.192880,0.412258,1
5044,(737),(Late),0.063800,0.6569,0.045841,0.718519,1.093802,1.0,0.003931,1.218909,0.091602,0.067927,0.179594,0.394151,1
7583,(MD82),(Late),0.039698,0.6569,0.027883,0.702381,1.069236,1.0,0.001805,1.152817,0.067430,0.041696,0.132559,0.372413,1
7746,(MD.88),(Late),0.015595,0.6569,0.011815,0.757576,1.153259,1.0,0.001570,1.415288,0.134998,0.017883,0.293430,0.387781,1


## **by Time**

In [201]:
times = set(df['scheduled_time_of_day'].unique())

In [202]:
late_rules_by_time = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (times, ))]
late_rules_by_time.shape

(1, 15)

In [203]:
late_rules_by_time

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
6942,(Evening),(Late),0.263233,0.6569,0.184783,0.701975,1.068618,1.0,0.011865,1.151246,0.087153,0.251285,0.131376,0.491635,1


Here are the key additional analyses you can provide to airlines:
1. Operational Performance Analytics

On-Time Performance (OTP) metrics by airline, route, and aircraft
Schedule reliability indicators
Aircraft utilization efficiency metrics

2. Delay Pattern Analysis

Time-based patterns: delays by hour, day of week, season
Holiday impact analysis: performance during special periods
Weekend vs weekday comparison

3. Predictive Delay Risk Scoring

Risk assessment for different operational scenarios
Probability-based scoring for delay likelihood
Early warning systems for high-risk flights

4. Resource Optimization

Gate/counter utilization analysis
Aircraft rotation efficiency
Peak hour capacity planning

5. Cost Impact Analysis

Financial impact of delays (estimated costs)
Monthly cost trends
ROI analysis for operational improvements

6. Competitive Benchmarking

Industry comparison metrics
Performance gaps identification
Market positioning analysis

7. Route Network Analysis

Route profitability assessment
Network optimization recommendations
Problematic routes identification

8. Passenger Impact Analysis

Severe delay analysis (>30 minutes)
Passenger disruption patterns
Service quality metrics

Key Benefits for Airlines:
Strategic Planning: Use seasonal and time-based patterns for capacity planning
Operational Efficiency: Identify bottlenecks and optimization opportunities
Cost Management: Quantify delay costs and prioritize improvements
Competitive Advantage: Benchmark against industry standards
Customer Experience: Reduce passenger disruptions and improve satisfaction
Risk Management: Proactive identification of high-risk scenarios
These analyses complement your association rule mining by providing quantitative metrics, predictive insights, and actionable recommendations that airlines can use for operational improvements and strategic decision-making.RetryClaude can make mistakes. Please double-check responses. Sonnet 4